## modelling

In [ ]:
!pip install pandas -q 
!pip install scikit-learn -q 

In [ ]:
# Import os module for operating system interactions like file path operations
import os

# Import time module for time-related functions and execution timing
import time

# Import psutil module for system monitoring and resource usage information
import psutil

# Import numpy library for numerical computations and array operations
import numpy as np

# Import pandas library for data manipulation and analysis with DataFrames
import pandas as pd

# Import LinearSVR for linear Support Vector Regression implementation
from sklearn.svm import LinearSVR

# Import KFold for k-fold cross-validation splitting strategy
from sklearn.model_selection import KFold

# Import mean_absolute_error for calculating mean absolute error regression metric
from sklearn.metrics import mean_absolute_error

# Import StandardScaler for feature standardization (zero mean, unit variance)
from sklearn.preprocessing import StandardScaler

# Import LinearRegression for ordinary least squares linear regression modeling
from sklearn.linear_model import LinearRegression

# Import pearsonr for computing Pearson correlation coefficient and p-value
from scipy.stats import pearsonr

# Import joblib for saving and loading Python objects (especially machine learning models)
import joblib

# Import Parallel and delayed for parallel computing and lazy function evaluation
from joblib import Parallel, delayed

In [ ]:
# Function to set low CPU priority for the current process
def set_low_cpu_priority():
    # Get current process ID
    pid = os.getpid()
    
    # Create Process object for the current process
    p = psutil.Process(pid)
    
    # Set process nice value to 19 (lowest priority on Linux/Unix systems)
    p.nice(19)

# Execute the function to set low CPU priority
set_low_cpu_priority()

In [ ]:
# Load training data from CSV file into pandas DataFrame
data = pd.read_csv('./input/train_baizhi_all1213.csv')

# Print column names of the loaded DataFrame for inspection
print(data.columns)

# Note: Based on actual requirements, modify the columns used for X features in subsequent code

In [ ]:
# Extract feature matrix X by removing specified columns and converting to numpy array
X = data.drop(columns=['age', 'sex',"Unnamed: 0","eid"]).values

# Extract target variable age as numpy array containing chronological ages
age = data['age'].values

# Extract sex as numpy array containing gender/sex information
sex = data['sex'].values

In [ ]:
# Gender filter: 0 = female, 1 = male
Gender = 0

# Filter feature matrix X to include only samples matching the specified gender
X = X[sex == Gender]

# Filter age array to include only samples matching the specified gender
age = age[sex == Gender]

In [ ]:
# Define number of folds for cross-validation (K=20 folds)
K = 20

# Initialize KFold cross-validator with shuffling and fixed random state
kf = KFold(n_splits=K, shuffle=True, random_state=42)

In [ ]:
# Pre-allocate array for storing predicted ages with zeros
age_predic = np.zeros(len(age))

In [ ]:
# Function to train model for each fold (runs in parallel)
def train_fold(k, train_index, test_index, X, age):
    """ Train K-fold cross-validation model (runs in parallel) """
    # Split data into training and testing sets for current fold
    xTrain, xTest = X[train_index], X[test_index]
    yTrain, yTest = age[train_index], age[test_index]

    # Standardize data using StandardScaler
    scaler = StandardScaler()
    xTrain_scaled = scaler.fit_transform(xTrain)
    xTest_scaled = scaler.transform(xTest)

    # Print progress message for current fold
    print(f'Training fold {k + 1}...')

    # Train Linear SVR model with specified parameters
    model = LinearSVR(C=1.0, epsilon=0.1, max_iter=10000)
    model.fit(xTrain_scaled, yTrain)

    # Predict test set using trained model
    yhat = model.predict(xTest_scaled)

    # Return test indices and corresponding predictions
    return test_index, yhat

# Execute 20-fold training in parallel
num_jobs = 4  # Number of CPU cores for parallel computation
results = Parallel(n_jobs=num_jobs)(
    delayed(train_fold)(k, train_index, test_index, X, age)
    for k, (train_index, test_index) in enumerate(kf.split(X))
)

# Merge results from parallel computation
for test_index, yhat in results:
    age_predic[test_index] = yhat

# Print completion message
print("K-fold Training completed!")

In [ ]:
# Calculate model prediction performance metrics
r_predic, _ = pearsonr(age, age_predic)  # Compute Pearson correlation coefficient
mae = mean_absolute_error(age, age_predic)  # Calculate Mean Absolute Error

# Print prediction performance results
print(f'Prediction outcome: correlation r={r_predic:.2f}, MAE={mae:.2f}')

In [ ]:
# Perform age bias correction
gap = age_predic - age  # Calculate prediction age bias (predicted - actual)

beta_model = LinearRegression()  # Use linear regression to correct age bias
beta_model.fit(age.reshape(-1, 1), gap)  # Fit linear model to predict bias from age

gap_corrected = gap - beta_model.predict(age.reshape(-1, 1))  # Compute corrected bias

In [ ]:
# Train final model on complete dataset
scaler_final = StandardScaler()
X_scaled = scaler_final.fit_transform(X)

final_model = LinearSVR(C=1.0, epsilon=0.1, max_iter=10000)
final_model.fit(X_scaled, age)

# Print completion message with sample size
print(f'Final model trained on full dataset (n={len(age)})')

In [ ]:
# Save model and results to file
save_path = f'model_sex{Gender}.pkl'

# Save model and related data using joblib
joblib.dump({'model': final_model, 
             'scaler': scaler_final,
             'age_real': age, 
             'age_predic': age_predic,
             'r_predic': r_predic,
             'mae': mae,
             'sex': Gender,
             'beta_model': beta_model}, save_path)

# Print confirmation message with save path
print(f'Model and results saved to {save_path}')

## test model

In [ ]:
# Import numpy library for numerical operations and array handling
import numpy as np

# Import scipy.io module for reading and writing MATLAB (.mat) files
import scipy.io as sio

# Import StandardScaler for standardizing features by removing mean and scaling to unit variance
from sklearn.preprocessing import StandardScaler

# Import mean_absolute_error for calculating mean absolute error regression loss
from sklearn.metrics import mean_absolute_error

# Import joblib for efficient serialization of Python objects (especially scikit-learn models)
import joblib

# Import os module for interacting with the operating system (file paths, directories, etc.)
import os

# Import time module for time-related functions and measuring execution time
import time

# Import psutil module for retrieving system information (CPU, memory, processes, etc.)
import psutil

# Note: numpy is imported again here (duplicate import - no effect on functionality)
import numpy as np

# Import pandas library for data manipulation and analysis with DataFrame structures
import pandas as pd

# Import LinearSVR for Linear Support Vector Regression implementation
from sklearn.svm import LinearSVR

# Import KFold for k-fold cross-validation splitting of datasets
from sklearn.model_selection import KFold

# Note: mean_absolute_error is imported again here (duplicate import - no effect on functionality)
from sklearn.metrics import mean_absolute_error

# Note: StandardScaler is imported again here (duplicate import - no effect on functionality)
from sklearn.preprocessing import StandardScaler

# Import LinearRegression for ordinary least squares linear regression
from sklearn.linear_model import LinearRegression

# Import pearsonr for calculating Pearson correlation coefficient and p-value
from scipy.stats import pearsonr

# Note: joblib is imported again here (duplicate import - no effect on functionality)
import joblib

# Import Parallel and delayed from joblib for parallel computing and lazy function execution
from joblib import Parallel, delayed

In [ ]:
# Load test data from CSV file into pandas DataFrame
data = pd.read_csv('./input/test_baizhi_all1213.csv')

# Print column names of the loaded DataFrame to inspect available features
print(data.columns)

# Note: Based on actual requirements, modify the columns used for X features in subsequent code

In [ ]:
# Extract feature matrix X_test by dropping specified columns and converting to numpy array
X_test = data.drop(columns=['age', 'sex','Unnamed: 0', 'eid']).values

# Extract target variable age_test as numpy array containing chronological ages
age_test = data['age'].values

# Extract sex_test as numpy array containing gender/sex information
sex_test = data['sex'].values

In [ ]:
# Select gender for testing
Gender = 0 # 0 for female, 1 for male
X_test = X_test[sex_test == Gender, :]
age_test = age_test[sex_test == Gender]

In [ ]:
# Load trained model and parameters
model_data = joblib.load(f'model_sex{Gender}.pkl')
model = model_data['model']
scaler = model_data['scaler']
beta_model = model_data['beta_model']

In [ ]:
# Standardize test data using training scaler
X_test_scaled = scaler.transform(X_test)

# Predict age
age_predic = model.predict(X_test_scaled)

In [ ]:
# Model performance
r_predic = np.corrcoef(age_predic, age_test)[0, 1]
mae = mean_absolute_error(age_test, age_predic)
print(f'Prediction outcome: correlation r={r_predic:.2f}, MAE={mae:.2f}')

In [ ]:
# Compute age gap
age_real = age_test
age_predic = age_predic
age_gap = age_predic - age_real

# Correct age gap using beta coefficients from training
beta_coef = beta_model.coef_[0]  
beta_intercept = beta_model.intercept_  
age_fit = beta_coef * age_real + beta_intercept
gap_resid = age_gap - age_fit

In [ ]:
# Save test results
joblib.dump({'age_real': age_real, 'age_predic': age_predic, 'r_predic': r_predic,
             'mae': mae, 'gap_resid': gap_resid}, f'test_sex{Gender}.pkl')

## calculate age

In [ ]:
# Import joblib for saving and loading Python objects, particularly machine learning models
import joblib

# Import numpy for numerical computations and array operations
import numpy as np

# Import os for operating system interactions like file path manipulation
import os

# Import time for time-related functions, often used for tracking execution time
import time

# Import psutil for system monitoring (CPU, memory, disk usage, etc.)
import psutil

# Note: numpy is imported again (duplicate import that has no functional impact)
import numpy as np

# Import pandas for data manipulation and analysis using DataFrames
import pandas as pd

# Import LinearSVR for linear Support Vector Regression implementation
from sklearn.svm import LinearSVR

# Import KFold for k-fold cross-validation data splitting strategy
from sklearn.model_selection import KFold

# Import mean_absolute_error for calculating MAE regression metric
from sklearn.metrics import mean_absolute_error

# Import StandardScaler for feature standardization (zero mean, unit variance)
from sklearn.preprocessing import StandardScaler

# Import LinearRegression for ordinary least squares linear regression modeling
from sklearn.linear_model import LinearRegression

# Import pearsonr for computing Pearson correlation coefficient and p-value
from scipy.stats import pearsonr

# Note: joblib is imported again (duplicate import that has no functional impact)
import joblib

# Import Parallel and delayed for parallel computing and lazy function evaluation
from joblib import Parallel, delayed

In [ ]:
# Load pre-trained model based on gender
Gender = 0  # 0 represents female, 1 represents male (gender classification)

# Load the model data from the saved file using joblib
model_data = joblib.load(f'model_sex{Gender}.pkl')

# Extract the trained SVM model from the loaded data
model = model_data['model']

# Extract the StandardScaler object used during training for feature standardization
scaler = model_data['scaler']

# Extract the linear regression model for age bias correction
beta_model = model_data['beta_model']

In [ ]:
# Load new sample data from CSV file (example data)
# X_new = np.load('your_new_data.npy')  # Should be in n_samples x n_features format

# Read CSV file containing new sample data into pandas DataFrame
data = pd.read_csv('./input/baizhi_all1213_female_all.csv')

# Print column names to inspect available features in the dataset
print(data.columns)

# Note: Based on actual requirements, modify the columns used for X features in subsequent code

In [ ]:
# Extract feature matrix X_new by removing specified columns and converting to numpy array
X_new = data.drop(columns=['Unnamed: 0', 'eid','sex', 'age']).values

# Apply feature scaling using the pre-trained StandardScaler to normalize new data
X_new_scaled = scaler.transform(X_new)

In [ ]:
# Perform brain age prediction using the trained model on scaled features
age_predic = model.predict(X_new_scaled)

# Extract age bias correction parameters from the linear regression model
beta_coef = beta_model.coef_[0]  # Slope coefficient for bias correction
beta_intercept = beta_model.intercept_  # Intercept for bias correction

# Calculate fitted age bias using linear correction formula
age_fit = beta_coef * age_predic + beta_intercept

# Compute bias-corrected brain age by subtracting predicted bias
age_corrected = age_predic - age_fit

# Print prediction results
print("Predicted Age:", age_predic)
print("Bias-Corrected Age:", age_corrected)

In [ ]:
# Add predicted and bias-corrected age columns to the original DataFrame
data['Predicted_Age'] = age_predic
data['Bias_Corrected_Age'] = age_corrected

# Import os module for operating system path operations
import os

# Define target output directory path 
output_directory = './output/'

# Construct full file path by joining directory and filename
output_filename = os.path.join(output_directory, 'baizhi_all1219_female.csv')

# Save the updated DataFrame with predictions to a new CSV file without index column
data.to_csv(output_filename, index=False)